<a href="https://colab.research.google.com/github/omri24/AudioGenAI/blob/Maya_DL/Efficientnet_Algo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cloning into 'AudioGenAI'...
remote: Repository not found.
fatal: repository 'https://github.com/mayamande/AudioGenAI.git/' not found


In [ ]:
!pip install mido
from google.colab import drive
drive.mount('/content/drive/')
%cd /content/drive/MyDrive/Colab Notebooks/project

import numpy as np
import torch
from torch import nn
from collections import OrderedDict
import torch.optim as optim
import math
import MIDI_DATASET
import MIDI_IO
import MIDI_coding


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.3 MB/s eta 0:00:00
Mounted at /content/drive/
/content/drive/MyDrive/Colab Notebooks/project


In [ ]:
print("GPU Available:", torch.cuda.is_available())


GPU Available: True


In [ ]:

"""
using as a start EfficientNet-B0 Architecture. The model consists of three parts:
1- stem layer - initial layer
2- body layer which consists of MBconv blocks
3- head - fully connected
"""
def calculate_output_size(input, kernel, padding, stride):
    output = (1 + (input - kernel + 2 * padding) / stride)
    if output < 1:
        raise ValueError(
            f"Invalid parameters: input={input}, kernel={kernel}, padding={padding}, stride={stride} result in negative or zero output size."
        )
    return math.floor(output)

class STEM_LAYER(nn.Module): #
    def __init__(self, input_size):
        super(STEM_LAYER, self).__init__()
        """
        stem layer class, architecture taken from https://medium.com/image-processing-with-python/efficientnetb0-architecture-stem-layer-496c7911a62d
        :param input_size: (in_channels, height, width)
        """
        out_channels=32
        kernel_size=(3,3)
        self.CONV1 = nn.Conv2d(kernel_size=kernel_size, in_channels=input_size[0], out_channels=out_channels, stride=2)
        self.output_size =(out_channels, calculate_output_size(input=input_size[1], kernel=kernel_size[1], padding=0, stride=2), calculate_output_size(input=input_size[2], kernel=kernel_size[0], padding=0, stride=2) )
        self.BN = nn.BatchNorm2d(num_features=  out_channels)
        self.SWISH = nn.SiLU()
    def forward(self, x):
        x=self.CONV1(x)
        x=self.BN(x)
        x=self.SWISH(x)
        return x

class MBCONV(nn.Module):
    def __init__(self,
                 input_size,
                 expand_ratio,
                 final_output_channels,
                 kernel_size,
                 stride):
        super(MBCONV, self).__init__()
        """
        :param input_size:  (input channels, height, width)
        :param expand_ratio:  for conv, if 1 skips expansion
        :param kernal_size: for conv
        :param stride: for conv

        Options to add -
        1 -  Squeeze-and-Excitation (SE) Block
        """
        self.padding=1
        #Expansion
        self.expand = (expand_ratio!=1)
        output_channels = input_size[0] * expand_ratio
        if(self.expand):
            self.conv_expan = nn.Conv2d(out_channels=output_channels, in_channels=input_size[0], kernel_size=kernel_size, stride=stride, padding=self.padding)
            output_size_expansion = (output_channels,
                                     calculate_output_size(input=input_size[1], kernel=kernel_size[0], padding=self.padding, stride=stride),
                                     calculate_output_size(input=input_size[2], kernel=kernel_size[1], padding=self.padding, stride=stride))
            self.BN_expan = nn.BatchNorm2d(num_features= output_channels)
            self.SWISH_expan = nn.SiLU()
        else:
            output_size_expansion=input_size
            self.conv_expan =None
            self.BN_expan = None
            self.SWISH_expan = None
        #Depth-wise Conv
        print(f"for depth-wise:{output_channels}, {kernel_size}, {stride}")
        self.DEPTH_CONV = nn.Conv2d(groups=output_channels,
                                    in_channels=output_channels,
                                    out_channels=output_channels,
                                    kernel_size=kernel_size,
                                    padding=self.padding,
                                    stride=stride)
        self.BN_depth=nn.BatchNorm2d(num_features=output_channels)
        self.SWISH_depth=nn.SiLU()
        #Projection Phase
        self.projection_conv = nn.Conv2d(in_channels=output_channels, out_channels=final_output_channels, kernel_size=(1,1), stride=stride, padding=self.padding)
        self.projection_bn=nn.BatchNorm2d(num_features=final_output_channels)
        self.Swish_projection=nn.SiLU()

        self.output_size = (final_output_channels, calculate_output_size(input=output_size_expansion[1], kernel=kernel_size[0], padding=self.padding, stride=stride), calculate_output_size(input=output_size_expansion[2], kernel=kernel_size[1], padding=self.padding, stride=stride))
    def forward(self, x):
        #expand
        if(self.expand):
            x = self.conv_expan(x)
            x= self.BN_expan(x)
            x=self.SWISH_expan(x)
        #depth-wise
        x=self.DEPTH_CONV(x)
        x=self.BN_depth(x)
        x=self.SWISH_depth(x)
        #projection
        x = self.projection_conv(x)
        x = self.projection_bn(x)
        x = self.Swish_projection(x)
        return x



class MODEL_BODY(nn.Module):
    def __init__(self, input_size, device):
        super(MODEL_BODY, self).__init__()
        """
        :param input_size: in_channels, height, width)
        """
        #mbconv_size = [(1,3,3,16), (6,3,3,24), (6,5,5,40),(6,3,3,80), (6, 5,5,112), (6,3,3,192), (6,3,3,320)] #(expand_ration, kernalsize, output_size)
        #number_of_layers=[1,2,2,3,3,4,1] #number of repeats

        mbconv_size = [(1,3,3,16), (3,3,3,24), (3,5,5,40)] #(expand_ration, kernalsize, output_size)
        number_of_layers=[1,1,1] #number of repeats
        self.conv_layers=[]
        for i,size in enumerate(mbconv_size):
            for n in range(number_of_layers[i]):
                new_layer=MBCONV(input_size=input_size, kernel_size=(size[1],size[2]), expand_ratio=size[0], stride=1, final_output_channels=size[3]).to(device)
                input_size=new_layer.output_size
                self.conv_layers.append(new_layer)
        self.output_size=input_size
    def forward(self,x):
        for l in self.conv_layers:
            x=l.forward(x)
        return x


class HEAD(nn.Module):
    def __init__(self,input_size, output_size):
        super(HEAD, self).__init__()

        """

        :param input_size:
        :param output_size:
        things to add:
        1 - dropout
        """
        self.conv = nn.Conv2d(in_channels=input_size[0], out_channels=128, kernel_size=(1,1), stride=1)
        self.bn = nn.BatchNorm2d(num_features=128)
        self.swish = nn.SiLU()
        self.flat = nn.Flatten()
        print(f"HEAD input_size: {input_size}")
        #self.fc = nn.Linear(in_features=1280*input_size[1]*input_size[2], out_features=output_size)
        self.fc = nn.Linear(in_features=128*1040, out_features=output_size*128)
        self.softmax =nn.Softmax(dim=0)
    def forward(self,x):
        x = self.conv(x)
        x=self.bn(x)
        x=self.swish(x)
        x=self.flat(x)
        x=self.fc(x)
        x=torch.reshape(x, (32*30,128))
        x=self.softmax (x)
        return x

class Model (nn.Module): #using
    def __init__(self, input_size, output_length, device):
        super(Model, self).__init__()

        """
        :param input_size: (in_channels, height, width)
        """
        self.stem_layer=STEM_LAYER(input_size=input_size).to(device)
        self.body = MODEL_BODY(input_size=self.stem_layer.output_size, device=device).to(device)
        self.head = HEAD(input_size=self.body.output_size, output_size=output_length).to(device) #midi notes
    def forward(self, x):
        x=self.stem_layer(x)
        x=self.body(x)
        x=self.head(x)
        return x


In [ ]:
class WeightedDistanceLoss(nn.Module):
    def __init__(self, weights=None):
        """
        :param weights: in tensor.torch format. matrix 128*128 for each note what is the wighted distance from a different note. The weight is multiplied by the distance.
        If the wight in type None, simple abs distance.
        """
        super(WeightedDistanceLoss, self).__init__()
        if (not weights):
            self.weights = torch.ones((128, 128))
        else:
            self.weights=weights

    def forward(self, predictions, targets):
        target_indices = targets.argmax(dim=1) # vector length batch_size*W
        predictions_indices = predictions.argmax(dim=1)# vector length batch_size*W
        distances = torch.mean(torch.square(predictions - targets), dim=1)# matrix length batch_size*L*W
        weight_values = self.weights[predictions_indices, target_indices] # matrix length batch_size*W
        weight_values.unsqueeze(-1) # Shape becomes (batch_size, W, 1)
        weighted_distances = torch.mul(distances, weight_values)
        print("finished calculating loss")
        return torch.sqrt(weighted_distances).mean()



In [ ]:
def train(dataset_test, dataset_train,x):
    """

    :param dataset: torch dataset
    :param x: length of matrix rows
    :return:
    things to add -
    1- validation parameters
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = Model((1,128,x), output_length=x, device=device)
    model.to(device)
    batch_size = 32
    epoch_size = 20
    loss_function = nn.NLLLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001) #TODO - add lr optimizers
    load_train = torch.utils.data.DataLoader(dataset=dataset_train, batch_size=batch_size, shuffle=True, drop_last=True)
    load_test = torch.utils.data.DataLoader(dataset=dataset_test, batch_size=batch_size, shuffle=False, drop_last=True)
    loss_array=[]
    for epoch in range(epoch_size):
        run_loss = 0
        test_loss = 0
        model.train()
        for X_train, t_train in load_train:
            X_train = X_train.unsqueeze(1).to(device)
            t_train = torch.argmax(t_train, dim=1).flatten().to(device)
            optimizer.zero_grad()
            y = model(X_train)
            loss = loss_function(y, t_train)
            loss.backward()
            optimizer.step()
            run_loss += loss.item()
            del y
        loss_array.append(run_loss)
        print(f"Loss: {run_loss} for epoch {epoch}")
        torch.save(model.state_dict(), f'model_weights_epoch{epoch+1}.pth')
    print(loss)

In [ ]:
dataset = MIDI_DATASET.load_dataset("/content/drive/MyDrive/Colab Notebooks/project/data_with_labels0.pth")
print("finished loading dataset")


/content/drive/MyDrive/Colab Notebooks/project/MIDI_DATASET.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(file_path)


finished loading dataset


In [ ]:
train_ds, validation_ds, test_ds = dataset.split_train_test(0.5, 0.2)


In [ ]:

torch.cuda.empty_cache()  # Free memory after each epoch
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
print(train_ds)
print("starting train")
train(dataset_test=test_ds, dataset_train=train_ds,x=30)

Allocated: 0.00 GB
Reserved: 0.00 GB
starting train
for depth-wise:32, (3, 3), 1
for depth-wise:48, (3, 3), 1
for depth-wise:72, (5, 5), 1
HEAD input_size: (40, 59, 10)
Loss: -6.916992087149993 for epoch 0
Loss: -7.26975445728749 for epoch 1
Loss: -7.419618187472224 for epoch 2
Loss: -7.521737870760262 for epoch 3
Loss: -7.51783156581223 for epoch 4
Loss: -7.645054934546351 for epoch 5
Loss: -7.522906004451215 for epoch 6
Loss: -7.662099597044289 for epoch 7
Loss: -7.707407069392502 for epoch 8
Loss: -7.710213964805007 for epoch 9
Loss: -7.7559136636555195 for epoch 10
Loss: -7.656021282076836 for epoch 11
Loss: -7.761232505552471 for epoch 12
Loss: -7.831785366870463 for epoch 13


KeyboardInterrupt: 

In [ ]:
def test(weights_path, test_input, x):
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model = Model((1,128,x), output_length=x, device=device)
  model.load_state_dict(torch.load(weights_path, map_location=device))
  model.eval()
  with torch.no_grad():  # Disable gradient computation for testing
      for x_test, t_test in test_input:
        x_test = x_test.unsqueeze(1).to(device)
        print(x_test.shape)
        output = model(x_test)
        return output, x_test



In [ ]:
def one_hot_encode_output(output_vector, file_name):
  out=output_vector[0].to("cpu").numpy()
  decoded_array=[]
  for a in out:
    argmax = np.argmax(a)
    one_hot_vect = [1 if argmax==i else 0 for i in range(128)]
    decoded_array.append(one_hot_vect)
  MIDI_IO.export_MIDI(np.array([decoded_array]), f"{file_name}.midi")

In [ ]:
epoch=11
batch_size = 32
load_test = torch.utils.data.DataLoader(dataset=test_ds, batch_size=batch_size, shuffle=False, drop_last=True)
output0, t0=test(f'model_weights_epoch{0}.pth', load_test ,x=30)
MIDI_IO.export_MIDI(t0[0], "no training")
output1, t0=test(f'model_weights_epoch{epoch}.pth', load_test ,x=30)


for depth-wise:32, (3, 3), 1
for depth-wise:48, (3, 3), 1
for depth-wise:72, (5, 5), 1
HEAD input_size: (40, 59, 10)


<ipython-input-6-752c72c375e0>:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path, map_location=device))


torch.Size([32, 1, 128, 30])
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])
MIDI file exported
for depth-wise:32, (3, 3), 1
for depth-wise:48, (3, 3), 1
for depth-wise:72, (5, 5), 1
HEAD input_size: (40, 59, 10)
torch.Size([32, 1, 128, 30])


In [ ]:
print(t0[0].shape)
one_hot_encode_output(output0.reshape(t0.shape)[0],  "one epoch")
one_hot_encode_output(output1.reshape(t0.shape)[0],  "ten epoch")

torch.Size([1, 128, 30])
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [1 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]]
MIDI file exported
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]]
MIDI file exported
